# LAB | Extractive Question Answering

This notebook demonstrates how Pinecone helps you build an extractive question-answering application. To build an extractive question-answering system, we need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A reader model to extract answers

We will use the SQuAD dataset, which consists of **questions** and **context** paragraphs containing question **answers**. We generate embeddings for the context passages using the retriever, index them in the vector database, and query with semantic search to retrieve the top k most relevant contexts containing potential answers to our question. We then use the reader model to extract the answers from the returned contexts.

Let's get started by installing the packages needed for notebook to run:

Install Dependencies

In [1]:
#!pip install -U langchain langchain-core langchain-classic langchain-pinecone langchain-huggingface datasets pinecone-client sentence-transformers torch

In [2]:
import getpass
import os

from pinecone import Pinecone, ServerlessSpec

# Clear PINECONE_API_KEY from environment to ensure re-prompting if needed
if "PINECONE_API_KEY" in os.environ:
    del os.environ["PINECONE_API_KEY"]

# Prompt for API key if not set
if not os.getenv("PINECONE_API_KEY"):
    api_key_input = getpass.getpass("Enter your Pinecone API key: ")
    os.environ["PINECONE_API_KEY"] = api_key_input

pinecone_api_key = os.environ.get("PINECONE_API_KEY")

# Verify the key is set (print masked version)
if pinecone_api_key:
    print(f"Pinecone API Key (masked): {pinecone_api_key[:4]}...{pinecone_api_key[-4:]}")
else:
    print("Pinecone API Key is not set.")

Enter your Pinecone API key: ··········
Pinecone API Key (masked): pcsk...HueD


# Load Dataset

Now let's load the SQUAD dataset from the HuggingFace Model Hub. We load the dataset into a pandas dataframe and filter the title, question, and context columns, and we drop any duplicate context passages.

In [75]:
from datasets import load_dataset

# load the squad dataset into a pandas dataframe
df = load_dataset("squad", split="train").to_pandas()

In [76]:
#print(df.head())
print(df.shape)

# select only title and context column
df = df[['title', 'question', 'context']]
# drop rows containing duplicate context passages
df = df.drop_duplicates(subset=['context'])

print(df.shape)
print(df.head())


(87599, 5)
(18891, 3)
                       title  \
0   University_of_Notre_Dame   
5   University_of_Notre_Dame   
10  University_of_Notre_Dame   
15  University_of_Notre_Dame   
20  University_of_Notre_Dame   

                                             question  \
0   To whom did the Virgin Mary allegedly appear i...   
5   When did the Scholastic Magazine of Notre dame...   
10  Where is the headquarters of the Congregation ...   
15  How many BS level degrees are offered in the C...   
20  What entity provides help with the management ...   

                                              context  
0   Architecturally, the school has a Catholic cha...  
5   As at most other universities, Notre Dame's st...  
10  The university is the major seat of the Congre...  
15  The College of Engineering was established in ...  
20  All of Notre Dame's undergraduate students are...  


# Initialize Pinecone Index

The Pinecone index stores vector representations of our context passages which we can retrieve using another vector (query vector). We first need to initialize our connection to Pinecone to create our vector index. For this, we need a free [API key]("https://app.pinecone.io/"), and then we initialize the connection like so:

In [77]:
from pinecone import ServerlessSpec

spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

# connect to pinecone environment
pc = Pinecone(api_key=pinecone_api_key, environment=spec.region)

Now we create a new index called "question-answering" — we can name the index anything we want. We specify the metric type as "cosine" and dimension as 384 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 384-dimension vectors.

In [78]:
index_name = "question-answering"

# Check if index exists & create with ServerlessSpec
if index_name not in pc.list_indexes().names():
    #create the index if it does not exist
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=spec
    )

# Connect to index
index = pc.Index(index_name)

# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all context passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will generate embeddings in a way that the questions and context passages containing answers to our questions are nearby in the vector space. We can use cosine similarity to calculate the similarity between the query and context embeddings to find the context passages that contain potential answers to our question.

We will use a SentenceTransformer model named ``multi-qa-MiniLM-L6-cos-v1`` designed for semantic search and trained on 215M (question, answer) pairs from diverse sources as our retriever.

In [7]:
#!pip install sentence-transformers torch langchain-huggingface langchain-text-splitters

In [79]:
import torch
from sentence_transformers import SentenceTransformer
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

# Set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# embeddings HuggingFace
embeds = HuggingFaceEmbeddings(model_name="sentence-transformers/multi-qa-MiniLM-L6-cos-v1")

# vectorStore
vectorstore = PineconeVectorStore(index=index, embedding=embeds, text_key="text")

# Load retriever model (multi-qa-MiniLM-L6-cos-v1 - 384 dimensions)
#retriever = None #use the 'multi-qa-MiniLM-L6-cos-v1' model from HuggingFace to build the retriever
retriever = vectorstore.as_retriever(search_type="similarity")

# Now retriever is loaded and ready
print("Retriever model loaded:", retriever)

Using device: cuda
Retriever model loaded: tags=['PineconeVectorStore', 'HuggingFaceEmbeddings'] vectorstore=<langchain_pinecone.vectorstores.PineconeVectorStore object at 0x78662dbf9c40> search_kwargs={}


# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, context passage, etc.

In [81]:
from tqdm.auto import tqdm
from uuid import uuid4

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=20,
)

texts = []
meta = []

# make chunks for each contexts
for i in tqdm(range(len(df))):
    chunks = text_splitter.split_text(df['context'].iloc[i])

    for j, chunk in enumerate(chunks):
        texts.append(chunk)
        meta.append({
            "chunk": j,
            "text": chunk,
            "title": df['title'].iloc[i]
        })

print("Total chunks:", len(texts))

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(texts), batch_size)):
    # extract batch
    batch_texts = texts[i:i+batch_size]
    # get metadata
    batch_meta = meta[i:i+batch_size]
    # generate embeddings for batch
    embeddings = embeds.embed_documents(batch_texts)
    # create unique IDs
    ids = [str(uuid4()) for _ in range(len(batch_texts))]
    # add all to upsert list
    to_upsert = zip(ids, embeddings, batch_meta)
    # upsert/insert these records to pinecone
    index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/18891 [00:00<?, ?it/s]

Total chunks: 44791


  0%|          | 0/700 [00:00<?, ?it/s]

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 44791}},
 'total_vector_count': 44791,
 'vector_type': 'dense'}

# Initialize Reader

We use the `deepset/electra-base-squad2` model from the HuggingFace model hub as our reader model. We load this model into a "question-answering" pipeline from HuggingFace transformers and feed it our questions and context passages individually. The model gives a prediction for each context we pass through the pipeline.

In [82]:
from transformers import pipeline

model_name = 'deepset/electra-base-squad2'
# load the reader model into a question-answering pipeline
reader = pipeline(tokenizer=model_name, model=model_name, task='question-answering', device=device)
reader

Device set to use cuda


Now all the components we need are ready. Let's write some helper functions to execute our queries. The `get_context` function retrieves the context embeddings containing answers to our question from the Pinecone index, and the `extract_answer` function extracts the answers from these context passages.

In [92]:
# gets context passages from the pinecone index
def get_context(question, top_k):
    # generate embeddings for the question
    xq = embeds.embed_query(question)
    #print(len(embeds.embed_documents(question)[0]))
    # search pinecone index for context passage with the answer
    xc = index.query(vector=xq, top_k=top_k, include_metadata=True)
    # extract the context passage from pinecone search result
    c = []
    for match in xc['matches']:
      c.append({
          "score": match['score'],
          "text": match['metadata'].get("text", ""),
          "title": match['metadata'].get("title", ""),
          "chunk": match['metadata'].get("chunk", None)
      })

    return c

In [93]:
from pprint import pprint

# extracts answer from the context passage
def extract_answer(question, context):
    results = []
    for c in context:
        # feed the reader the question and contexts to extract answers
        answer = reader(question=question, context=c["text"])
        # add the context to answer dict for printing both together
        answer["context"] = c
        results.append(answer)
    # sort the result based on the score from reader model
    sorted_result = pprint(sorted(results, key=lambda x: x['score'], reverse=True))
    return sorted_result


In [94]:
question = "How much oil is Egypt producing in a day?"
context = get_context(question, top_k = 3)
context

[{'score': 0.766763687,
  'text': 'Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of',
  'title': 'Egypt',
  'chunk': 0.0},
 {'score': 0.714068413,
  'text': 'Egypt has a developed energy market based on coal, oil, natural gas, and hydro power. Substantial coal deposits in the northeast Sinai are mined at the rate of about 600,000 tonnes (590,000 long tons; 660,000 short tons) per year. Oil and gas are produced in the western desert regions, the Gulf of Suez, and the Nile Delta. Egypt has huge reserves of gas, estimated at 2,180 cubic kilometres (520 cu',
  'title': 'Egypt',
  'chunk': 0.0},
 {'score': 0.708340645,
  'text': 'additional gas 

In [95]:
#for c in context:
#    print(c["score"])
#    print(c["text"][:300])
#    print("------")

0.766763687
Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consu
------
0.714068413
Egypt has a developed energy market based on coal, oil, natural gas, and hydro power. Substantial coal deposits in the northeast Sinai are mined at the rate of about 600,000 tonnes (590,000 long tons; 660,000 short tons) per year. Oil and gas are produced in the western desert regions, the Gulf of S
------
0.708340645
additional gas volumes in summer, while encouraging factories to plan their annual maintenance for those months of peak demand, said EGPC chairman, Tarek El Barkatawy. Egypt produces its own energy, but has been a net oil importer since 2008 and is rapidly becoming a net importer of natural gas.
------


As we can see, the retiever is working fine and gets us the context passage that contains the answer to our question. Now let's use the reader to extract the exact answer from the context passage.

In [97]:
extract_answer(question, context)

[{'answer': '691,000 bbl/d',
  'context': {'chunk': 0.0,
              'score': 0.766763687,
              'text': 'Egypt was producing 691,000 bbl/d of oil and 2,141.05 '
                      'Tcf of natural gas (in 2013), which makes Egypt as the '
                      'largest oil producer not member of the Organization of '
                      'the Petroleum Exporting Countries (OPEC) and the '
                      'second-largest dry natural gas producer in Africa. In '
                      '2013, Egypt was the largest consumer of oil and natural '
                      'gas in Africa, as more than 20% of total oil '
                      'consumption and more than 40% of',
              'title': 'Egypt'},
  'end': 33,
  'score': 0.9999773184665628,
  'start': 20},
 {'answer': '600,000 tonnes',
  'context': {'chunk': 0.0,
              'score': 0.714068413,
              'text': 'Egypt has a developed energy market based on coal, oil, '
                      'natural gas, an

The reader model predicted with 99% accuracy the correct answer *691,000 bbl/d* as seen from the context passage. Let's run few more queries.

In [98]:
question = "What are the first names of the men that invented youtube?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'answer': 'Hurley and Chen',
  'context': {'chunk': 0.0,
              'score': 0.567344725,
              'text': 'According to a story that has often been repeated in '
                      'the media, Hurley and Chen developed the idea for '
                      'YouTube during the early months of 2005, after they had '
                      'experienced difficulty sharing videos that had been '
                      "shot at a dinner party at Chen's apartment in San "
                      'Francisco. Karim did not attend the party and denied '
                      'that it had occurred, but Chen commented that the idea '
                      'that YouTube was founded after',
              'title': 'YouTube'},
  'end': 79,
  'score': 0.9999470710754395,
  'start': 64}]


In [99]:
question = "What is Albert Eistein famous for?"
context = get_context(question, top_k=1)
extract_answer(question, context)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[{'answer': 'his theories of special relativity and general relativity',
  'context': {'chunk': 0.0,
              'score': 0.491976291,
              'text': 'Albert Einstein is known for his theories of special '
                      'relativity and general relativity. He also made '
                      'important contributions to statistical mechanics, '
                      'especially his mathematical treatment of Brownian '
                      'motion, his resolution of the paradox of specific '
                      'heats, and his connection of fluctuations and '
                      'dissipation. Despite his reservations about its '
                      'interpretation, Einstein also made contributions to',
              'title': 'Modern_history'},
  'end': 86,
  'score': 0.9796618223190308,
  'start': 29}]


Let's run another question. This time for top 3 context passages from the retriever.

In [104]:
question = "Who was the first person to step foot on the moon?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': 'Armstrong',
  'context': {'chunk': 1.0,
              'score': 0.742717743,
              'text': 'downrange error, Armstrong took over manual flight '
                      'control at about 180 meters (590 ft), and guided the '
                      'Lunar Module to a safe landing spot at 20:18:04 UTC, '
                      'July 20, 1969 (3:17:04 pm CDT). The first humans on the '
                      'Moon would wait another six hours before they ventured '
                      'out of their craft. At 02:56 UTC, July 21 (9:56 pm CDT '
                      'July 20), Armstrong became the first human to set foot '
                      'on the Moon.',
              'title': 'Space_Race'},
  'end': 26,
  'score': 0.9991099834442139,
  'start': 17},
 {'answer': 'Aldrin',
  'context': {'chunk': 0.0,
              'score': 0.578810692,
              'text': 'The first step was witnessed by at least one-fifth of '
                      'the population of Earth, or about 

The result looks pretty good.

In [74]:
#pc.delete_index(index_name)

### Add a few more questions. What did you observe?

In [105]:
question = "How many days are in year 2024?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': '365.2425',
  'context': {'chunk': 0.0,
              'score': 0.577417433,
              'text': 'In addition to the change in the mean length of the '
                      'calendar year from 365.25 days (365 days 6 hours) to '
                      '365.2425 days (365 days 5 hours 49 minutes 12 seconds), '
                      'a reduction of 10 minutes 48 seconds per year, the '
                      'Gregorian calendar reform also dealt with the '
                      'accumulated difference between these lengths. The '
                      'canonical Easter tables were devised at the end of the '
                      'third century, when the vernal',
              'title': 'Gregorian_calendar'},
  'end': 113,
  'score': 4.422098797363105e-06,
  'start': 105},
 {'answer': '30 to 90 days',
  'context': {'chunk': 2.0,
              'score': 0.564576209,
              'text': 'of 30 to 90 days begins.',
              'title': 'Endangered_Species_Act'},
  'end': 16,
 

In [106]:
question = "What is the most famous spring flower?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': 'Irish shamrock',
  'context': {'chunk': 1.0,
              'score': 0.515201092,
              'text': 'Tudor rose; Scots thistle; Welsh leek; Irish shamrock; '
                      'Australian wattle; Canadian maple leaf; New Zealand '
                      'silver fern; South African protea; lotus flowers for '
                      "India and Ceylon; and Pakistan's wheat, cotton, and "
                      'jute.',
              'title': 'Elizabeth_II'},
  'end': 53,
  'score': 0.0009796425001695752,
  'start': 39},
 {'answer': 'Gentians',
  'context': {'chunk': 1.0,
              'score': 0.490311623,
              'text': "torch-like with the smoking blueness of Pluto's "
                      'gloom." Gentians tend to "appear" repeatedly as the '
                      'spring blooming takes place at progressively later '
                      'dates, moving from the lower altitude to the higher '
                      'altitude meadows where the snow melts much lat